### Databricks Mosaic AI Vector Index Creation

![vector_index_creation](./Assets/vector_index_creation.png)

### Installing Utilities and Libraries

In [ ]:
%pip install databricks-vectorsearch==0.63

### Restarting the Python Kernel

In [ ]:
dbutils.library.restartPython()

### Enabling CDC (Change Data Capture) on the Final RAG Dataset in Unity Catalog

In [ ]:
# Enable change data feed for the existing Delta table
spark.sql("""
ALTER TABLE RAG.final_rag_dataset
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

### Creating the Vector Index and Endpoint

In [ ]:
from databricks.vector_search.client import VectorSearchClient

vector_client = VectorSearchClient()

# To recreate the endpoint after deletion:
vector_client.create_endpoint(
     name="vector_search_endpoint",
     endpoint_type="STANDARD"
 )


index = vector_client.create_delta_sync_index(
   endpoint_name="vector_search_endpoint",
   source_table_name="YOUR_UC_NAME.SCHEMA.TABLE_NAME",
   index_name="YOUR_UC_NAME.SCHEMA.rag_vector_index",
   pipeline_type="TRIGGERED",
   primary_key="id",
   embedding_source_column="chunk",
   embedding_model_endpoint_name="databricks-gte-large-en"
  )

### Triggering our Index - Information Retrieval

#### Querying the Vector Index with a Sample Question

In [ ]:
user_question = "can you tell me what hotels are offered by Margies Travel in Dubai?"

results_dict = index.similarity_search(
            query_text = user_question,
            columns = ["content_path","chunk"],
            num_results=10,
            query_type="hybrid"
          )

for content in results_dict['result']['data_array']:
    print(json.dumps(content, indent=2, ensure_ascii=False))